In [1]:
import os
import shutil
import glob
from pathlib import Path
from tqdm import tqdm

import pandas as pd
pd.set_option('display.max_columns', 500)
import geopandas as gpd 
from shapely import Point


In [2]:
dir_co2 = "D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2"
dir_ch4 = "D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CH4"

In [3]:
if not os.path.exists(os.path.join(dir_co2, 'filtered_data')):
    os.makedirs(os.path.join(dir_co2, 'filtered_data'))

if not os.path.exists(os.path.join(dir_ch4, 'filtered_data')):
    os.makedirs(os.path.join(dir_ch4, 'filtered_data'))

sectors = glob.glob(os.path.join(dir_co2, 'DATA/*'))
sectors = [Path(i).name for i in sectors]

for sector in sectors:
    print(sector)
    files_co2 = glob.glob(os.path.join(dir_co2, 'DATA', sector, '*emissions_sources.csv'))
    for f in files_co2:
        try:
            shutil.copy(f, os.path.join(dir_co2, 'filtered_data'))
            print(f"Copied {f} to {os.path.join(dir_co2, 'filtered_data')}")
        except Exception as e:
            print(f"Failed to copy {f} to {os.path.join(dir_co2, 'filtered_data')}")
            continue
    

    files_ch4 = glob.glob(os.path.join(dir_ch4, 'DATA', sector, '*emissions_sources.csv'))
    for f in files_ch4:
        try:
            shutil.copy(f, os.path.join(dir_ch4, 'filtered_data'))
            print(f"Copied {f} to {os.path.join(dir_ch4, 'filtered_data')}")
        except Exception as e:
            print(f"Failed to copy {f} to {os.path.join(dir_ch4, 'filtered_data')}")
            continue

agriculture
Copied D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\DATA\agriculture\cropland-fires_emissions_sources.csv to D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\filtered_data
Copied D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\DATA\agriculture\enteric-fermentation-cattle-pasture_emissions_sources.csv to D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\filtered_data
Copied D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\DATA\agriculture\manure-left-on-pasture-cattle_emissions_sources.csv to D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\filtered_data
Copied D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\DATA\agriculture\rice-cultivation_emissions_sources.csv to D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\filtered_data
Copied D:/Work/WB/LDT/countries/SRB/raw_data/climate_trace_SRB_CO2\DATA\agriculture\synthetic-fertilizer-application_emissions_sources.csv to D:/Work/WB/LDT/c

In [4]:
co2_csv_paths = glob.glob(os.path.join(dir_co2, 'filtered_data', '*.csv'))
dfs_co2 = []
for p in tqdm(co2_csv_paths):
    dfs_co2.append(pd.read_csv(p))
df_co2 = pd.concat(dfs_co2, axis=0).reset_index(drop=True)    

ch4_csv_paths = glob.glob(os.path.join(dir_ch4, 'filtered_data', '*.csv'))
dfs_ch4 = []
for p in tqdm(ch4_csv_paths):
    dfs_ch4.append(pd.read_csv(p))
df_ch4 = pd.concat(dfs_ch4, axis=0).reset_index(drop=True)  

100%|██████████| 37/37 [00:00<00:00, 40.79it/s]


In [5]:
df_co2['start_time'] = pd.to_datetime(df_co2['start_time'], errors='coerce')
df_co2['end_time'] = pd.to_datetime(df_co2['end_time'], errors='coerce')
df_co2['year'] = df_co2['start_time'].dt.year
df_co2 = df_co2[['source_id', 'source_name', 'source_type', 'sector', 'subsector', 'start_time', 'end_time', 'year', 'lat', 'lon', 'gas', 'emissions_quantity', 'emissions_factor', 'emissions_factor_units']]
df_co2['geometry'] = df_co2.apply(lambda row: Point((row['lon'], row['lat'])), axis=1)
df_co2 = gpd.GeoDataFrame(df_co2, geometry='geometry', crs="EPSG:4326")


df_ch4['start_time'] = pd.to_datetime(df_ch4['start_time'], errors='coerce')
df_ch4['end_time'] = pd.to_datetime(df_ch4['end_time'], errors='coerce')
df_ch4['year'] = df_ch4['start_time'].dt.year
df_ch4 = df_ch4[['source_id', 'source_name', 'source_type', 'sector', 'subsector', 'start_time', 'end_time', 'year', 'lat', 'lon', 'gas', 'emissions_quantity', 'emissions_factor', 'emissions_factor_units']]
df_ch4['geometry'] = df_ch4.apply(lambda row: Point((row['lon'], row['lat'])), axis=1)
df_ch4 = gpd.GeoDataFrame(df_ch4, geometry='geometry', crs="EPSG:4326")


In [6]:
#Read Shape File --> The shape file gives a MultiPolygon Geometry Column
gdf = gpd.read_file(r"D:\Work\WB\LDT\countries\SRB\shapefiles\gadm41_SRB_2.json")

#Adjust for GeoSpatial Data
center = gpd.GeoDataFrame(gdf[['GID_2', 'NAME_2']])

#Change the MultiPolygon Geometry Column to make it more useful
center['geometry'] = gdf.centroid
center = center.to_crs(gdf.crs)
center['lat'] = center.geometry.y
center['lon'] = center.geometry.x

C:\Users\sonle\AppData\Local\Temp\ipykernel_36584\910939600.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center['geometry'] = gdf.centroid
C:\Users\sonle\AppData\Local\Temp\ipykernel_36584\910939600.py:8: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  center['geometry'] = gdf.centroid


In [7]:
emissions_co2 = gpd.sjoin(df_co2, gdf, predicate = 'within',
                      how = 'inner')
grouped_emissions_co2 = emissions_co2.groupby(['GID_2', 'NAME_2', 'year']).agg({
    'emissions_quantity': 'sum',        # Sum of the 'Sales' column
    'emissions_factor': 'mean'     # Average of the 'Quantity' column
}).reset_index()

emissions_ch4 = gpd.sjoin(df_ch4, gdf, predicate = 'within',
                      how = 'inner')
grouped_emissions_ch4 = emissions_ch4.groupby(['GID_2', 'NAME_2', 'year']).agg({
    'emissions_quantity': 'sum',        # Sum of the 'Sales' column
    'emissions_factor': 'mean'     # Average of the 'Quantity' column
}).reset_index()

In [ ]:
coal_emissions = emissions_co2[emissions_co2['source_type'] == 'coal']
grouped_coal_emissions = coal_emissions.groupby(['GID_2', 'NAME_2', 'year']).agg({
    'emissions_quantity': 'sum'
}).reset_index()
grouped_coal_emissions = grouped_coal_emissions.rename({'emissions_quantity': 'total_powerplant_coal_emissions_quantity'}, axis=1)
grouped_coal_emissions

KeyError: "Column(s) ['emissions'] do not exist"

In [30]:
grouped_coal_emissions.to_csv("D:/Work/WB/LDT/countries/SRB/datasets/SRB_co2e_emissions_coal_full.csv", index=False)

In [33]:
grouped_emissions_co2_sector = emissions_co2.groupby(['GID_2', 'NAME_2', 'year', 'sector']).agg({
    'emissions_quantity': 'sum',        # Sum of the 'Sales' column 
}).reset_index()

grouped_emissions_co2_sector = grouped_emissions_co2_sector.pivot_table(index=['GID_2', 'NAME_2', 'year'], columns=['sector'], values=['emissions_quantity'], aggfunc='sum').reset_index()
grouped_emissions_co2_sector.columns = [
    f"{col[1]}_emissions" if col[0] == "emissions_quantity" else col[0] 
    for col in grouped_emissions_co2_sector.columns.ravel()
]
grouped_emissions_co2_sector = grouped_emissions_co2_sector.fillna(0)
grouped_emissions_co2_sector.to_csv("D:/Work/WB/LDT/countries/SRB/datasets/SRB_co2e_emissions_sector_full.csv", index=False)

In [ ]:
grouped_emissions_co2.to_csv("D:/Work/WB/LDT/countries/SRB/datasets/SRB_co2e_emissions_full.csv", index=False)

grouped_emissions_ch4.to_csv("D:/Work/WB/LDT/countries/SRB/datasets/SRB_ch4_emissions_full.csv", index=False)


In [87]:
coal_emissions = emissions_co2[(emissions_co2['sector'] == 'power') & (emissions_co2['source_type'] == 'coal')]

grouped_coal_emissions = coal_emissions.groupby(['GID_2', 'NAME_2', 'year']).agg({
    'emissions_quantity': 'sum',        # Sum of the 'Sales' column
    'emissions_factor': 'mean'     # Average of the 'Quantity' column
}).reset_index()

grouped_coal_emissions = grouped_coal_emissions.rename({"emissions_quantity": "total_powerplant_coal_emissions_quantity", "emissions_factor": "total_powerplant_coal_emissions_factor"}, axis=1)

grouped_coal_emissions.to_csv("D:/Work/WB/LDT/countries/SRB/datasets/SRB_emissions_from_coal_full.csv", index=False)